# Round 2 Algorithmic Trading Strategy

This notebook documents the active `trader.py` strategy for `ASH_COATED_OSMIUM` and `INTARIAN_PEPPER_ROOT`. The design goal is maximum expected XIREC profit with explicit controls against regime breaks, stale books, overfit prediction, and inventory concentration.


## Executive thesis

The active strategy replays at **249284.0 XIRECs** across days -1, 0, and 1, versus **246681.0 XIRECs** from the archived round 1 parameters. The latest iteration adds **+611.0 XIRECs** over the previously pushed Round 2 strategy while reducing terminal osmium inventory.

The largest edge remains structural: `INTARIAN_PEPPER_ROOT` behaves like a nearly deterministic upward-drifting claim. `ASH_COATED_OSMIUM` is stationary around 10,000 with strong one-step mean reversion, so it is traded as an inventory-limited market-making and mean-reversion book.


## Market Access Fee bid

`trader.py` bids **825 XIRECs** for extra market access. This is a measured middle bid: it is far above public example/default-style bids, but it does not sacrifice a large fraction of the round's expected PnL if accepted.

Volume-sensitivity replay now shows the tuned strategy improves as extra access arrives, mostly through more profitable osmium mean-reversion fills.

| volume scenario | combined PnL | osmium PnL | min day | end osmium inventory |
| --- | --- | --- | --- | --- |
| 0.8 | 247023.0 | 8851.0 | 81914.0 | [69, 55, 59] |
| 1.0 | 249284.0 | 11094.0 | 82641.0 | [69, 60, 63] |
| 1.25 | 251851.0 | 13648.0 | 83608.0 | [65, 62, 63] |


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
diagnostics = json.loads((ROOT / 'logs' / 'round2_diagnostics.json').read_text())
diagnostics['deterministic_backtest']['combined_pnl']


## Data regime

Rows with `mid_price == 0` have empty visible top-of-book data and are excluded from slope and training calculations.

| day | product | first | last | slope / 100 | resid sigma | ret ac1 | empty rows |
| --- | --- | --- | --- | --- | --- | --- | --- |
| -1 | ASH_COATED_OSMIUM | 9991.0 | 10002.0 | -0.000053 | 4.464 | -0.506 | 15 |
| -1 | INTARIAN_PEPPER_ROOT | 11001.5 | 11999.5 | 0.100000 | 2.194 | -0.498 | 13 |
| 0 | ASH_COATED_OSMIUM | 10003.0 | 10008.0 | 0.000157 | 5.641 | -0.506 | 16 |
| 0 | INTARIAN_PEPPER_ROOT | 11998.5 | 13000.0 | 0.099998 | 2.364 | -0.489 | 18 |
| 1 | ASH_COATED_OSMIUM | 10008.0 | 9993.0 | -0.000713 | 4.578 | -0.491 | 22 |
| 1 | INTARIAN_PEPPER_ROOT | 13000.0 | 13999.5 | 0.100004 | 2.543 | -0.508 | 16 |

Pepper root has an OLS slope almost exactly `0.001` per timestamp, or `0.1` per 100 timestamps, on every day. Osmium has near-zero trend and residual standard deviation around 4.5 to 5.6 XIRECs, making it a much smaller and more mean-reverting process.


## Contingent-claims framing

I treat each filled unit as a short-horizon claim on terminal marked value. For a long unit, payoff is approximately `terminal_mid - fill_price`; for a short unit, payoff is `fill_price - terminal_mid`.

| product | min long payoff | mean long payoff | max long payoff | interpretation |
| --- | --- | --- | --- | --- |
| ASH_COATED_OSMIUM | -23.0 | -8.0 | 2.0 | mean-reversion claim |
| INTARIAN_PEPPER_ROOT | 990.5 | 992.3 | 994.0 | trend claim |

A first-ask pepper long pays between 990.5 and 994.0 XIRECs per unit historically. A first-ask osmium long is not attractive by itself, so osmium trades only when cheap or rich versus a fair value estimate.


## Statistical and ML screens

I trained simple next-tick models on days -1 and 0, then held out day 1. Features were EMA deviation, best-level imbalance, spread, and normalized timestamp. Ridge and KNN both confirm the same usable structure: short-horizon mean reversion plus order-book pressure.

| product | ridge MSE | baseline MSE | ridge direction | KNN MSE | KNN direction |
| --- | --- | --- | --- | --- | --- |
| ASH_COATED_OSMIUM | 7.219 | 8.571 | 69.6% | 7.641 | 67.7% |
| INTARIAN_PEPPER_ROOT | 6.797 | 8.515 | 72.0% | 7.295 | 70.4% |

The predictor itself is not shipped directly. A direct ridge fair-value injection made execution too jumpy in replay. The deployed version uses the robust ingredients only: EMA-style fair smoothing, capped imbalance influence, passive inventory skew, and active crossing inventory skew.


## Latest execution iteration

The previous strategy only skewed passive osmium quotes for inventory. The new strategy also skews active crossing thresholds:

```python
cross_skew = cross_inventory_skew * position / position_limit
buy_limit = fair - take_edge - cross_skew
sell_limit = fair + take_edge - cross_skew
```

When long, this makes further buys harder and sells easier. When short, it makes buys easier and further sells harder. Lowering osmium `imbalance_weight` from `0.8` to `0.4` keeps imbalance as a small nudge instead of letting it overpower the inventory control.

Selected osmium params: `fair_alpha=0.12`, `imbalance_weight=0.4`, `take_edge=0.0`, `make_edge=3.0`, `cross_inventory_skew=1.0`.


## Strategy logic

`INTARIAN_PEPPER_ROOT`:

1. Estimate live intercept from volume-weighted top-of-book fair value minus `0.001 * timestamp`.
2. Maintain fair as `intercept + 0.001 * timestamp`.
3. Buy asks up to `fair + 8`, capped by `max_take = 10`, until the 80-unit limit is reached.
4. Place passive bids slightly below fair, skewed upward while inventory is below target.
5. If observed intercept falls more than 35 XIRECs below day-open intercept, stop adding risk and flatten.

`ASH_COATED_OSMIUM`:

1. Estimate fair from volume-weighted best bid/ask, blended with recent trades when present.
2. Smooth fair with `fair_alpha = 0.12` and add `0.4 * imbalance`.
3. Cross cheap asks/rich bids with active inventory skew so the strategy naturally reduces crowded inventory.
4. Add symmetric passive quotes around fair with separate passive inventory skew.
5. If book fair moves more than 35 XIRECs from the 10,000 anchor, stop fresh entries and flatten inventory.


## Backtest result

| day | total PnL | pepper PnL | osmium PnL | end pepper | end osmium |
| --- | --- | --- | --- | --- | --- |
| -1 | 82856.0 | 79427.0 | 3429.0 | 80 | 69 |
| 0 | 83787.0 | 79399.0 | 4388.0 | 80 | 60 |
| 1 | 82641.0 | 79364.0 | 3277.0 | 80 | 63 |

Combined deterministic replay: **249284.0 XIRECs**. Product split: **238190.0** from pepper root and **11094.0** from osmium. Improvement versus archived round 1 parameters: **2603.0 XIRECs**.


## Risk and stress testing

Monte Carlo execution stress uses cross-fill probability `0.97`, up to `1` XIREC adverse slippage, and `3.0` XIRECs of closing mark noise. Across `40` draws, the PnL summary is:

| min | p05 | median | mean | p95 | max | Pr >= 245k |
| --- | --- | --- | --- | --- | --- | --- |
| 244731.8 | 245755.6 | 246458.7 | 246412.0 | 247156.5 | 247514.1 | 97.5% |

The main residual risk is a pepper regime break. The guard compares live intercept against day-open intercept, so normal timestamp drift does not trip the stop. The new osmium crossing skew reduces risk by making active orders inventory-aware instead of only quote-aware.


In [ ]:
# Re-run diagnostics after changing trader.py by regenerating logs/round2_diagnostics.json.
diagnostics['selected_parameters']
